## 📦 Install Dependencies

In [ ]:
!pip install plotly folium networkx numpy pandas ipywidgets

## 🗂️ Generate Synthetic Anchor Dataset

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import uuid

np.random.seed(42)

cities = {
    "Lagos": (6.5244, 3.3792),
    "Nairobi": (-1.2921, 36.8219),
    "London": (51.5074, -0.1278),
    "São Paulo": (-23.5505, -46.6333),
}

categories = ["food", "art", "nature", "business", "education"]
handles = [f"@user_{i}" for i in range(1, 51)]

rows = []
base_date = datetime(2024, 1, 1)

for city_name, (base_lat, base_lng) in cities.items():
    for _ in range(50):
        rows.append({
            "id": str(uuid.uuid4()),
            "lat": base_lat + np.random.uniform(-0.05, 0.05),
            "lng": base_lng + np.random.uniform(-0.05, 0.05),
            "category": np.random.choice(categories),
            "reaction_count": np.random.randint(1, 501),
            "created_at": base_date + timedelta(days=int(np.random.uniform(0, 30))),
            "creator_handle": np.random.choice(handles),
            "city": city_name,
        })

df = pd.DataFrame(rows)
df.head(10)

## 🗺️ 2D Folium Map — Clustered Markers

In [ ]:
import folium
from folium.plugins import MarkerCluster

m = folium.Map(location=[10, 10], zoom_start=2)

color_map = {
    "food": "red",
    "art": "purple",
    "nature": "green",
    "business": "blue",
    "education": "orange",
}

marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    radius = max(3, min(20, row["reaction_count"] / 50))
    popup_html = (
        f"<b>{row['creator_handle']}</b><br>"
        f"Category: {row['category']}<br>"
        f"Reactions: {row['reaction_count']}"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=radius,
        color=color_map[row["category"]],
        fill=True,
        fill_color=color_map[row["category"]],
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=200),
    ).add_to(marker_cluster)

m

## 🌐 3D Plotly Scatter — Anchor Distribution

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

discrete_color_map = {
    "food": "#EF553B",
    "art": "#AB63FA",
    "nature": "#00CC96",
    "business": "#636EFA",
    "education": "#FFA15A",
}

fig = px.scatter_3d(
    df,
    x="lng",
    y="lat",
    z="reaction_count",
    color="category",
    color_discrete_map=discrete_color_map,
    size="reaction_count",
    size_max=18,
    hover_data=["id", "city", "category", "creator_handle", "reaction_count", "created_at"],
    template="plotly_dark",
    title="Spatial Canvas \u2014 3D Anchor Distribution",
)

fig.update_layout(
    scene_camera=dict(eye=dict(x=1.5, y=1.5, z=1.5)),
)

fig.show()

## 🎬 Animated 3D — Anchor Appearance Over Time

In [ ]:
df_sorted = df.sort_values("created_at").copy()
df_sorted["date_str"] = df_sorted["created_at"].dt.strftime("%Y-%m-%d")

# Build cumulative frames: each date frame includes all anchors up to that date
unique_dates = sorted(df_sorted["date_str"].unique())
cumulative_frames = []
for date in unique_dates:
    frame_df = df_sorted[df_sorted["date_str"] <= date].copy()
    frame_df["frame"] = date
    cumulative_frames.append(frame_df)

df_animated = pd.concat(cumulative_frames, ignore_index=True)

fig_anim = px.scatter_3d(
    df_animated,
    x="lng",
    y="lat",
    z="reaction_count",
    color="category",
    animation_frame="frame",
    template="plotly_dark",
    title="Anchor Appearance Over Time",
)

fig_anim.show()

## 📊 Analytics Charts

In [ ]:
from plotly.subplots import make_subplots

# 1. Bar chart: anchors per city
city_counts = df["city"].value_counts().reset_index()
city_counts.columns = ["city", "count"]

fig_bar = px.bar(
    city_counts,
    x="city",
    y="count",
    template="plotly_dark",
    title="Anchors per City",
    color="city",
)
fig_bar.show()

# 2. Pie chart: category distribution
cat_counts = df["category"].value_counts().reset_index()
cat_counts.columns = ["category", "count"]

fig_pie = px.pie(
    cat_counts,
    names="category",
    values="count",
    template="plotly_dark",
    title="Category Distribution",
    color="category",
    color_discrete_map=discrete_color_map,
)
fig_pie.show()

# 3. Time-series line chart: cumulative creation rate
df_time = df.copy()
df_time["date"] = df_time["created_at"].dt.date
daily_counts = df_time.groupby("date").size().reset_index(name="count")
daily_counts = daily_counts.sort_values("date")
daily_counts["cumulative"] = daily_counts["count"].cumsum()

fig_line = px.line(
    daily_counts,
    x="date",
    y="cumulative",
    template="plotly_dark",
    title="Cumulative Anchor Creation Over Time",
    markers=True,
)
fig_line.show()

## 🎛️ Sandbox — Interactive Filters

In [ ]:
from ipywidgets import IntSlider, Dropdown, interact

print("🎛️ Sandbox \u2014 Interactive Filters")

@interact(
    radius_km=IntSlider(min=1, max=50, value=25, description="Radius (km)"),
    min_reactions=IntSlider(min=0, max=100, value=0, description="Min Reactions"),
    city=Dropdown(options=["All", "Lagos", "Nairobi", "London", "S\u00e3o Paulo"], value="All", description="City"),
)
def sandbox_filter(radius_km, min_reactions, city):
    city_centers = {
        "Lagos": (6.5244, 3.3792),
        "Nairobi": (-1.2921, 36.8219),
        "London": (51.5074, -0.1278),
        "S\u00e3o Paulo": (-23.5505, -46.6333),
    }

    def haversine_km(lat1, lng1, lat2, lng2):
        R = 6371
        dlat = np.radians(lat2 - lat1)
        dlng = np.radians(lng2 - lng1)
        a = np.sin(dlat / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlng / 2) ** 2
        return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    filtered = df[df["reaction_count"] >= min_reactions].copy()
    if city != "All":
        filtered = filtered[filtered["city"] == city]

    # Apply radius filter: keep anchors within radius_km of their city center
    mask = filtered.apply(
        lambda r: haversine_km(r["lat"], r["lng"], *city_centers[r["city"]]) <= radius_km, axis=1
    )
    filtered = filtered[mask]

    # Filtered 3D scatter
    fig_filtered = px.scatter_3d(
        filtered,
        x="lng",
        y="lat",
        z="reaction_count",
        color="category",
        color_discrete_map=discrete_color_map,
        size="reaction_count",
        size_max=18,
        template="plotly_dark",
        title=f"Filtered Anchors (radius={radius_km}km, min_reactions={min_reactions}, city={city})",
    )
    fig_filtered.show()

    # Filtered bar chart: category counts
    cat_data = filtered["category"].value_counts().reset_index()
    cat_data.columns = ["category", "count"]
    fig_cat = px.bar(
        cat_data,
        x="category",
        y="count",
        color="category",
        color_discrete_map=discrete_color_map,
        template="plotly_dark",
        title="Filtered Category Counts",
    )
    fig_cat.show()

## 💾 Export

In [ ]:
import os

export_dir = "exports"
os.makedirs(export_dir, exist_ok=True)

export_path = os.path.join(export_dir, "spatial_canvas_3d.html")
fig.write_html(export_path)

print(f"\u2705 Exported 3D scatter to {export_path}")